
# 01 – Financial Data Preparation & Baseline Modelling

This notebook prepares the **financial dataset** and estimates **baseline ARX / VAR models** without any emotion-based variables.

It:

1. Loads raw OHLCV financial data (Bitcoin, Gold, Crude Oil, S&P 500, FX, 10Y yield).
2. Cleans and aligns the series on a common daily index.
3. Constructs **log returns** and **control variables** (FX, oil, yield).
4. Performs basic exploratory analysis and stationarity checks.
5. Estimates baseline **ARX** (per asset) and a multivariate **VAR** model.
6. Exports a tidy dataset for use in the emotion/EPI notebook.

> **Note:** You may need to adjust file paths and column names to match your actual data.


In [ ]:

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.api import VAR
import statsmodels.api as sm

# Display and plotting options
pd.options.display.float_format = "{:.4f}".format
plt.rcParams["figure.figsize"] = (10, 4)

DATA_DIR = "./data/"
OUTPUT_DIR = "./data/"

RAW_FINANCIAL_FILE = DATA_DIR + "financial_raw.json"  # change to your actual file
FINANCIAL_PANEL_FILE = OUTPUT_DIR + "financial_returns_and_controls.parquet"
FINANCIAL_PANEL_CSV = OUTPUT_DIR + "financial_returns_and_controls.csv"



## 1. Load raw financial data

Expected structure (example):

```json
{
  "Date_": "2024-11-19 00:00:00",
  "Close_BTC-USD": ...,
  "High_BTC-USD": ...,
  "Low_BTC-USD": ...,
  "Open_BTC-USD": ...,
  "Volume_BTC-USD": ...,
  ...
}
```


In [ ]:

# --- Load raw data ---
# If your file is CSV, use read_csv instead.
try:
    df_raw = pd.read_json(RAW_FINANCIAL_FILE)
except ValueError:
    df_raw = pd.read_csv(RAW_FINANCIAL_FILE)

# Basic cleaning: parse dates and sort
if "Date_" in df_raw.columns:
    df_raw["Date_"] = pd.to_datetime(df_raw["Date_"])
    df_raw = df_raw.sort_values("Date_").set_index("Date_")
else:
    # Adjust here if your date column has a different name
    raise KeyError("Expected a 'Date_' column. Please adjust the code to your date column name.")

print(df_raw.head())
print(df_raw.info())



## 2. Sanity checks & renaming

We rename columns to simpler names (e.g. `close_btc`, `close_gold`, etc.) so they are easier to work with.


In [ ]:

# Inspect available columns
df_raw.columns


In [ ]:

# Define a renaming map; extend / adjust as needed
rename_map = {
    "Close_BTC-USD": "close_btc",
    "Open_BTC-USD": "open_btc",
    "High_BTC-USD": "high_btc",
    "Low_BTC-USD": "low_btc",
    "Volume_BTC-USD": "vol_btc",

    "Close_GC=F": "close_gold",
    "Open_GC=F": "open_gold",
    "High_GC=F": "high_gold",
    "Low_GC=F": "low_gold",
    "Volume_GC=F": "vol_gold",

    "Close_CL=F": "close_oil",
    "Open_CL=F": "open_oil",
    "High_CL=F": "high_oil",
    "Low_CL=F": "low_oil",
    "Volume_CL=F": "vol_oil",

    "Close_^GSPC": "close_spx",
    "Open_^GSPC": "open_spx",
    "High_^GSPC": "high_spx",
    "Low_^GSPC": "low_spx",
    "Volume_^GSPC": "vol_spx",

    "Close_CNY=X": "close_usdcny",
    "Open_CNY=X": "open_usdcny",
    "High_CNY=X": "high_usdcny",
    "Low_CNY=X": "low_usdcny",
    "Volume_CNY=X": "vol_usdcny",

    "Close_EUR=X": "close_eurusd",
    "Open_EUR=X": "open_eurusd",
    "High_EUR=X": "high_eurusd",
    "Low_EUR=X": "low_eurusd",
    "Volume_EUR=X": "vol_eurusd",

    "Close_^TNX": "close_ust10y",
    "Open_^TNX": "open_ust10y",
    "High_^TNX": "high_ust10y",
    "Low_^TNX": "low_ust10y",
    "Volume_^TNX": "vol_ust10y",
}

df = df_raw.rename(columns=rename_map)

print("After renaming:")
print(df.head())


## 3. Align frequencies and handle non-trading days

We keep the native trading-day calendar (no synthetic weekend/holiday rows). Prices are forward-filled within existing rows only, and a flag marks rows that needed filling.

In [ ]:

# Create a continuous business-day index over the sample
full_index = pd.date_range(df.index.min(), df.index.max(), freq="B")
df = df.reindex(full_index)

# Identify price and volume columns
price_cols = [c for c in df.columns if c.startswith(("close_", "open_", "high_", "low_"))]
volume_cols = [c for c in df.columns if c.startswith("vol_")]

# Forward-fill prices, leave volumes as-is
df[price_cols] = df[price_cols].ffill()

# Optional: create flags indicating forward-filled rows
df["ffill_flag"] = df[price_cols].isna().any(axis=1)

print(df.head())



## 4. Construct returns and control variables

We compute:

- **Log returns** for Bitcoin, Gold, S&P 500.
- **Log differences** for FX and oil.
- **First difference** for the 10-year yield.


In [ ]:

def log_return(series):
    return np.log(series).diff()

returns = pd.DataFrame(index=df.index)

# Asset returns
returns["r_btc"] = log_return(df["close_btc"])
returns["r_gold"] = log_return(df["close_gold"])
returns["r_spx"] = log_return(df["close_spx"])

# Control variables
returns["dln_eurusd"] = log_return(df["close_eurusd"])   # USD/EUR
returns["dln_usdcny"] = log_return(df["close_usdcny"])   # USD/CNY
returns["dln_oil"] = log_return(df["close_oil"])
returns["dy10"] = df["close_ust10y"].diff()

# Drop initial NaNs from differencing
returns = returns.dropna()

returns.head()



## 5. Exploratory data analysis

Time-series plots, summary statistics, and correlations.


In [ ]:

# Plot returns
for col in ["r_btc", "r_gold", "r_spx"]:
    ax = returns[col].plot(title=f"{col} – daily log returns")
    ax.set_xlabel("Date")
    ax.set_ylabel("Return")
    plt.show()


In [ ]:

# Summary statistics
returns.describe().T


In [ ]:

# Rolling 30-day volatility for selected assets
rolling_window = 30
vol = returns[["r_btc", "r_gold", "r_spx"]].rolling(rolling_window).std()

ax = vol.plot(title=f"{rolling_window}-day rolling volatility")
ax.set_xlabel("Date")
ax.set_ylabel("Volatility (SD)")
plt.show()


In [ ]:

# Correlation matrix
corr = returns[["r_btc", "r_gold", "r_spx"]].corr()
corr



## 6. Stationarity checks (ADF tests)


In [ ]:

def adf_test(series, name=""):
    result = adfuller(series.dropna(), autolag="AIC")
    print(f"ADF Test for {name}")
    print(f"  Test statistic: {result[0]:.4f}")
    print(f"  p-value:        {result[1]:.4f}")
    print("  Critical values:")
    for k, v in result[4].items():
        print(f"    {k}: {v:.4f}")
    print("-" * 40)

for col in returns.columns:
    adf_test(returns[col], name=col)



## 7. Baseline ARX models (no emotions)

We estimate an ARX model for each asset:

\[ r_{i,t} = \alpha + \sum_{k=1}^{p} \phi_k r_{i,t-k} + \beta' x_t + \varepsilon_t, \]

where \(x_t\) are the macro control variables.


In [ ]:

# Prepare a working copy
data = returns.copy()

# Choose AR order p
p = 1

# Create lags for each asset
for asset in ["r_btc", "r_gold", "r_spx"]:
    for lag in range(1, p + 1):
        data[f"{asset}_l{lag}"] = data[asset].shift(lag)

# Control variable names
control_cols = ["dln_eurusd", "dln_usdcny", "dln_oil", "dy10"]

data_arx = data.dropna()
data_arx.head()


In [ ]:

def fit_arx(data, asset, p=1, control_cols=None):
    if control_cols is None:
        control_cols = []

    cols_lag = [f"{asset}_l{lag}" for lag in range(1, p + 1)]
    cols_exog = cols_lag + control_cols

    df = data.dropna(subset=[asset] + cols_exog)
    y = df[asset]
    X = sm.add_constant(df[cols_exog])

    model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags":5})
    return model

arx_results = {}
for asset in ["r_btc", "r_gold", "r_spx"]:
    model = fit_arx(data, asset, p=p, control_cols=control_cols)
    arx_results[asset] = model
    print(f"\n=== ARX results for {asset} ===")
    print(model.summary().tables[1])



## 8. Baseline VAR model (no emotions)

We estimate a VAR model on the three return series, optionally with exogenous macro controls.


In [ ]:

# VAR with exogenous controls
Y = returns[["r_btc", "r_gold", "r_spx"]]
X_exog = returns[control_cols]

# Align and drop missing rows
var_data = pd.concat([Y, X_exog], axis=1).dropna()
Y_var = var_data[["r_btc", "r_gold", "r_spx"]]
X_var = var_data[control_cols]

# Select lag order (can inspect select_order results)
var_model = VAR(Y_var)
lag_order_results = var_model.select_order(maxlags=10)
print(lag_order_results.summary())


In [ ]:

# Fit VAR with chosen lag order (e.g. 1)
p_var = 1
var_model = VAR(Y_var)
var_results = var_model.fit(maxlags=p_var, trend="c", exog=X_var)

print(var_results.summary())



## 9. (Optional) Event calendar hooks

Here we show a template for adding tariff-event dummies. Adjust dates to match your actual event list.


In [ ]:

# Example: create a simple event calendar (replace with real dates)
event_dates = [
    "2025-02-10",  # Initial tariff announcement
    "2025-03-15",  # Another key event
    "2025-04-20",  # Retaliatory tariffs
]

events = pd.DataFrame({
    "event_date": pd.to_datetime(event_dates),
    "event_dummy": 1
}).set_index("event_date")

# Merge into returns as a dummy column
returns_events = returns.join(events["event_dummy"], how="left")
returns_events["event_dummy"] = returns_events["event_dummy"].fillna(0)

returns_events["event_dummy"].value_counts()



## 10. Export tidy dataset

We export a panel containing the returns and control variables for reuse in the emotion/EPI notebook.


In [ ]:

panel = returns.copy()

# Optionally include event dummies or other flags
# panel = returns_events

panel.to_parquet(FINANCIAL_PANEL_FILE)
panel.to_csv(FINANCIAL_PANEL_CSV)

print("Saved financial panel to:")
print("  ", FINANCIAL_PANEL_FILE)
print("  ", FINANCIAL_PANEL_CSV)



### Notebook 1 complete ✅

You now have a cleaned financial dataset with returns and control variables saved to disk.
The next notebook will load this panel and merge it with the Emotion and EPI indices.
